In [ ]:
import numpy as np

# ============================
# 1. Hitung Gini Impurity
# ============================
def gini(y):
    """Hitung Gini impurity untuk label y"""
    m = len(y)
    if m == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / m
    return 1 - np.sum(probs**2)

# ============================
# 2. Cari split terbaik
# ============================
def best_split(X, y):
    """Cari feature & threshold terbaik berdasarkan information gain"""
    m, n_features = X.shape
    best_gain = 0
    best_feature, best_threshold = None, None

    parent_impurity = gini(y)

    for feature in range(n_features):
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            # Bagi dataset
            left_idx = np.where(X[:, feature] <= threshold)[0]
            right_idx = np.where(X[:, feature] > threshold)[0]

            if len(left_idx) == 0 or len(right_idx) == 0:
                continue

            # Hitung impurity anak
            n = len(y)
            n_left, n_right = len(left_idx), len(right_idx)
            child_impurity = (n_left / n) * gini(y[left_idx]) + \
                             (n_right / n) * gini(y[right_idx])

            # Information Gain
            gain = parent_impurity - child_impurity

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gain

# ============================
# 3. Node class
# ============================
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

# ============================
# 4. Decision Tree
# ============================
class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._grow(X, y, depth=0)

    def _grow(self, X, y, depth):
        num_samples, num_features = X.shape
        num_labels = len(np.unique(y))

        # Kondisi berhenti
        if (depth >= self.max_depth
            or num_labels == 1
            or num_samples < self.min_samples_split):
            leaf_value = self._most_common(y)
            return Node(value=leaf_value)

        # Cari split terbaik
        feature, threshold, gain = best_split(X, y)

        if gain == 0:
            leaf_value = self._most_common(y)
            return Node(value=leaf_value)

        # Split data
        left_idx = np.where(X[:, feature] <= threshold)[0]
        right_idx = np.where(X[:, feature] > threshold)[0]

        left_child = self._grow(X[left_idx, :], y[left_idx], depth+1)
        right_child = self._grow(X[right_idx, :], y[right_idx], depth+1)

        return Node(feature, threshold, left_child, right_child)

    def _most_common(self, y):
        values, counts = np.unique(y, return_counts=True)
        return values[np.argmax(counts)]

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

# ============================
# 5. Contoh penggunaan
# ============================
if __name__ == "__main__":
    # Dataset sederhana
    X = np.array([
        [2, 3],
        [1, 1],
        [3, 2],
        [6, 5],
        [7, 8],
        [8, 6]
    ])
    y = np.array([0, 0, 0, 1, 1, 1])

    # Training
    clf = DecisionTree(max_depth=3)
    clf.fit(X, y)

    # Prediksi
    X_test = np.array([[5, 4], [1, 2]])
    print("Prediksi:", clf.predict(X_test))
